<a href="https://colab.research.google.com/github/aaronzr/aaron-nnrr/blob/nate_steps/Huggin_MultiStep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "transformers<5" -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "tomg-group-umd/huginn-0125",
    trust_remote_code=True,
    device_map="auto",
    dtype=torch.bfloat16 # Updated from torch_dtype to dtype
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

raven_config_minimal.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/tomg-group-umd/huginn-0125:
- raven_config_minimal.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!


raven_modeling_minimal.py: 0.00B [00:00, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.77G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.74G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.38G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/178 [00:00<?, ?B/s]

In [ ]:
n = 20          # number of examples to run
#num_steps = 32  # Huginn's recurrent depth (more = more test-time compute)
step_counts = [16, 8, 4]    # recurrent iterations per token to compare
VERBOSE = True

In [ ]:
import re
from datasets import load_dataset
from transformers import AutoTokenizer, GenerationConfig

tokenizer = AutoTokenizer.from_pretrained("tomg-group-umd/huginn-0125")

# Load GSM8K and keep only the first n examples
ds = load_dataset("openai/gsm8k", "main", split="test").select(range(n))

gen_config = GenerationConfig(
    max_new_tokens=512,
    stop_strings=["<|end_text|>", "<|end_turn|>"],
    use_cache=True,
    do_sample=False,
    temperature=None, top_k=None, top_p=None, min_p=None,
    return_dict_in_generate=True,
    eos_token_id=65505,
    bos_token_id=65504,
    pad_token_id=65509,
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [ ]:
def extract_number(text):
    """Return the last number in a string, stripped of commas/$."""
    nums = re.findall(r"-?\d[\d,]*\.?\d*", text)
    return nums[-1].replace(",", "").rstrip(".") if nums else None

In [ ]:
def is_correct(pred, gold):
    try:
        return pred is not None and float(pred) == float(gold)
    except ValueError:
        return False

In [ ]:
def sync():
    # GPU work is asynchronous, so sync before reading the clock
    if torch.cuda.is_available():
        torch.cuda.synchronize()

In [ ]:
def build_input_ids(question):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": question + "\nThink step by step, then give the final answer as a number."},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return tokenizer.encode(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)

In [ ]:
# Warm-up so CUDA init / kernel compilation doesn't skew the first timed example
_ = model.generate(build_input_ids(ds[0]["question"]), gen_config,
                   tokenizer=tokenizer, num_steps=step_counts[0])

In [ ]:
from timeit import default_timer as timer
import statistics
sync()
all_stats = {}

for num_steps in step_counts:
    print("\n" + "#" * 80)
    print(f"# RUNNING WITH num_steps = {num_steps}")
    print("#" * 80)

    correct_flags, times, token_counts = [], [], []

    for i, ex in enumerate(ds):
        input_ids = build_input_ids(ex["question"])

        sync()
        start = timer()
        out = model.generate(input_ids, gen_config, tokenizer=tokenizer, num_steps=num_steps)
        sync()
        elapsed = timer() - start

        new_tokens = out.sequences[0, input_ids.shape[1]:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True)

        gold = ex["answer"].split("####")[-1].strip().replace(",", "")
        pred = extract_number(response)
        correct = is_correct(pred, gold)

        correct_flags.append(correct)
        times.append(elapsed)
        token_counts.append(len(new_tokens))

        if VERBOSE:
            print("=" * 80)
            print(f"[steps={num_steps}] Example {i + 1}/{n}")
            print(f"INPUT:\n{ex['question']}\n")
            print(f"RECURRENT STEPS: {num_steps} per token "
                  f"({len(new_tokens)} tokens generated, ~{num_steps * len(new_tokens)} core-block passes total)")
            print(f"OUTPUT:\n{response}\n")
            print(f"GOLD: {gold} | PRED: {pred} | {'✓ CORRECT' if correct else '✗ WRONG'}")
            print(f"TIME: {elapsed:.2f}s ({len(new_tokens) / elapsed:.1f} tokens/s)")

    all_stats[num_steps] = {
        "accuracy": sum(correct_flags) / n,
        "n_correct": sum(correct_flags),
        "mean": statistics.mean(times),
        "median": statistics.median(times),
        "stdev": statistics.stdev(times) if n > 1 else 0.0,
        "min": min(times),
        "max": max(times),
        "total": sum(times),
        "mean_tokens": statistics.mean(token_counts),
        "tok_per_s": sum(token_counts) / sum(times),
    }

# ---- Summary ----
print("\n" + "=" * 100)
print(f"SUMMARY ({n} examples per setting)")
print("=" * 100)
header = (f"{'steps':>5} | {'acc':>12} | {'mean(s)':>8} | {'median':>7} | {'stdev':>6} | "
          f"{'min':>6} | {'max':>6} | {'total(s)':>8} | {'avg tok':>7} | {'tok/s':>6}")
print(header)
print("-" * len(header))
for steps, s in all_stats.items():
    print(f"{steps:>5} | {s['n_correct']:>3}/{n} ({s['accuracy']:>5.1%}) | {s['mean']:>8.2f} | "
          f"{s['median']:>7.2f} | {s['stdev']:>6.2f} | {s['min']:>6.2f} | {s['max']:>6.2f} | "
          f"{s['total']:>8.2f} | {s['mean_tokens']:>7.1f} | {s['tok_per_s']:>6.1f}")


################################################################################
# RUNNING WITH num_steps = 16
################################################################################
[steps=16] Example 1/20
INPUT:
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

RECURRENT STEPS: 16 per token (265 tokens generated, ~4240 core-block passes total)
OUTPUT:
To calculate Janet's daily earnings from selling fresh duck eggs at the farmers' market, we need to follow these steps:

1. Determine the total number of eggs laid by the ducks each day.
2. Subtract the number of eggs eaten for breakfast and baked muffins.
3. Subtract the number of eggs sold at the market.
4. Divide the remaining number of eggs by the price per egg.

Step 1: 16 eggs per day
Step 2: 